# Parallel Simulation of Long Return Periods

Bridge load assessment often needs very long simulations - 100 years of traffic or more - which can take many hours on one core. 
PyBTLS can split one logical simulation into independent day-chunks that run in parallel across your CPU cores, then merge the results back into a single output, as if the simulation had been run in one piece. 

All you do is pass `no_chunk` to `add_sim()`. PyBTLS handles the per-chunk random seeding, the output directories, and the exact merging of every output type. 

To do this, we need to: 

**Step 1:** Define a bridge and a traffic generator (as usual); 

**Step 2:** Request the outputs; 

**Step 3:** Add the simulation with `no_chunk` and run it on several cores; 

**Step 4:** Read the merged results; 

**Step 5:** Save and reload the outputs. 

See the *Parallel Simulation Guide* in the documentation for the statistical rationale, the per-output merge semantics, and measured speedups.

---
Import all the necessary modules.

In [ ]:
import pybtls as pb
from pathlib import Path
import shutil

shutil.rmtree("./temp_parallel", ignore_errors=True)  # clean re-runs

---
**Step 1:** Define a bridge with one load effect (mid-span bending moment of a 30 m simply supported beam) and a matching 2-lane traffic generator.

In [ ]:
inf_line = pb.InfluenceLine(IL_type="built-in")
inf_line.set_IL(id=1, length=30.0)  # mid-span moment

bridge = pb.Bridge(length=30.0, no_lane=2)
bridge.add_load_effect(inf_line_surf=inf_line, threshold=100.0)

In [ ]:
traffic_gen = pb.TrafficGenerator(no_lane=2)

for lane_index, lane_dir in [(1, 1), (2, 2)]:
    lfc = pb.LaneFlowComposition(lane_index=lane_index, lane_dir=lane_dir)
    lfc.assign_lane_data(  # 24 hours per day by default
        hourly_truck_flow=[120] * 24,
        hourly_car_flow=[20] * 24,
        hourly_speed_mean=[80 / 3.6 * 10] * 24,  # in dm/s
        hourly_speed_std=[10 / 3.6 * 10] * 24,  # in dm/s
        hourly_truck_composition=[[23, 2.8, 31.7, 42.5]] * 24,
    )
    traffic_gen.add_lane(
        vehicle_gen=pb.VehicleGenGrave(traffic_site="Auxerre"),
        headway_gen=pb.HeadwayGenFreeflow(),
        lfc=lfc,
    )

traffic_gen.set_start_time(0.0)

---
**Step 2:** Request the outputs. Every output type can be merged across chunks, so ask for whatever your analysis needs.

In [ ]:
output_config = pb.OutputConfig()
output_config.set_BM_output(write_vehicle=True, write_summary=True, write_mixed=True)
output_config.set_POT_output(write_vehicle=True, write_summary=True, write_counter=True)
output_config.set_stats_output(
    write_flow_stats=True, write_overall=True, write_intervals=True
)
output_config.set_fatigue_output(write_rainflow_output=True)

---
**Step 3:** Add the simulation with `no_chunk` and run it. 

Here a 12-day simulation is split into 4 chunks of 3 days, run on 4 cores. For a real study you might use `no_day = 25000, no_chunk = 25` on a many-core machine. 

Rules checked by `add_sim` (a `ValueError` explains any violation): 
- the traffic must come from a `TrafficGenerator` (recorded traffic cannot be re-seeded); 
- `no_day` must divide evenly by `no_chunk`; 
- the chunk length must align with the configured block-maximum / POT-counter block sizes and statistics interval (any whole-day chunk works with the defaults). 

The `seed` is the master seed: chunk `i` runs with `seed + i`, so the same `seed` and `no_chunk` always reproduce the same results.

In [ ]:
sim_task = pb.Simulation(output_dir=Path("./temp_parallel"))
sim_task.add_sim(
    bridge=bridge,
    traffic=traffic_gen,
    no_day=12,
    output_config=output_config,
    time_step=0.1,
    min_gvw=35,
    tag="parallel_demo",
    seed=42,
    no_chunk=4,
)
sim_task.run(no_core=4)

---
**Step 4:** Read the merged results. 

`get_output()` returns a single merged view per simulation - the same interface as an unchunked run. Block and interval indices continue seamlessly across the chunk boundaries.

In [ ]:
output = sim_task.get_output()["parallel_demo"]

print("available outputs:", output.get_summary())
print("chunks:", output.no_chunk, "| master seed:", output.master_seed)

In [ ]:
bm_summary = output.read_data("BM_summary")
for name, data in bm_summary.items():
    print(name)
    print(data.head(12))  # daily block maxima, blocks 1..12 across all chunks

In [ ]:
pot_counter = output.read_data("POT_counter")
for name, data in pot_counter.items():
    print(name)
    print(data)

---
The per-chunk results remain on disk under `parallel_demo/chunk_000` ... `chunk_003` and can be inspected individually, e.g. for convergence checks.

In [ ]:
first_chunk_bm = output.read_chunk_data("BM_summary")[0]
for name, data in first_chunk_bm.items():
    print(name)
    print(data)  # 3 daily blocks of the first chunk only

---
**Step 5:** Save and reload the outputs. 

`save_output` writes a small human-readable JSON manifest (the data itself stays in the simulation output files), which can be reloaded in a later session with `load_output`.

In [ ]:
pb.save_output(sim_task.get_output(), file_path=Path("./temp_parallel/outputs.json"))

reloaded = pb.load_output(file_path=Path("./temp_parallel/outputs.json"))
print(reloaded["parallel_demo"].read_data("E_cumulative_statistics"))

---
**Where to go next** 

- The *Parallel Simulation Guide* page covers why day-chunking is statistically valid, what "merged exactly" means for each output type, and the measured speedups. 
- The *Block-Maxima* and *Peak-Over-Threshold* tutorials show the extreme value analysis you would run on these merged outputs.